In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import warnings
import os

import gfdl_utils.core as gu
import CM4Xutils
import cftime
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xhistogram

plt.rcParams.update({'font.size': 11})

In [43]:
include_piControl=False

### Load postprocessed model datasets

In [44]:
from common import *
grids = load_datasets()

Inferring Z grid coordinate: depth `z_`
Inferring Z grid coordinate: depth `z_`


In [45]:
dlat = 1.
lat_bins = np.arange(-90., 90+dlat, dlat)
lat_centers = np.arange(-90.+dlat/2, 90+dlat/2, dlat)

for sim in ["CM4Xp25_forced", "CM4Xp125_forced"]:
    ds = grids[sim]._ds.sel(year=slice(1991,1997))
    ds_reduced = xr.Dataset(coords=ds.coords)
    for tr in ["cfc11", "cfc12", "sf6"]:
        ds_reduced[rf"{tr}_vertical_int"] = (ds.cfc11*ds.thkcello*m2_per_km2).mean("year").sum("z_l")
        
        volcello = ds.thkcello.fillna(0.) * ds.areacello.fillna(0.)
        cfc11_content_zonally_integrated = xhistogram.xarray.histogram(
            ds.geolat.rename("lat"),
            bins=[lat_bins],
            dim=("xh", "yh", "year"),
            weights=ds[tr].fillna(0.)*volcello,
            bin_dim_suffix="",
            block_size=None,
            keep_coords=True
        )
        volume_content_zonally_integrated = xhistogram.xarray.histogram(
            ds.geolat.rename("lat"),
            bins=[lat_bins],
            dim=("xh", "yh", "year"),
            weights=volcello,
            bin_dim_suffix="",
            block_size=None,
            keep_coords=True,
        )
        ds_reduced[rf"{tr}_zonal_int"] = cfc11_content_zonally_integrated/volume_content_zonally_integrated
    
    ds_reduced.to_netcdf(f"../data/processed/tracer_integrals_{sim}.nc", mode="w")